# 海馬迴 MRI 影像組學輔助內側顳葉癲癇側化（Colab 分析流程 v2）

對應計畫書第五段「分析流程」與切分圖。**順序是本 notebook 最重要的設計：**

| 模組 | 計畫書步驟 | 內容 | 是否使用跨受試者統計量 |
|---|---|---|---|
| M0 | — | 環境、參數、版本紀錄 | 否 |
| M1 | 步驟 1 | 整理世代、套用納入排除、**先切分**（70/30，seed 2026），測試集另存 | 否（只分層抽樣） |
| M2 | 步驟 0 | 逐一受試者前處理：N4、重新取樣、個人腦內 z 標準化 | 否 |
| M3 | 步驟 0 | PyRadiomics 萃取（原始／侵蝕／膨脹三版本遮罩）、左右不對稱指標 | 否 |
| M4 | 步驟 2–4 | **只用訓練集**：摺內 ComBat → ICC → 冗餘移除 → 標準化 → LASSO（≤ 5）→ 擬合；選超參數、定 Youden 切點、定版 | 是，但全部在摺內 |
| M5 | 步驟 5–6 | **保留測試集只開啟一次**：AUC、AUPRC、校準、決策曲線、模型 C vs A 的 ΔAUC | — |

**兩種模式**：`DEMO_MODE = True`（預設）時，M3 以 PyRadiomics 內建示範影像驗證萃取可執行，M4–M5 以模擬特徵表演練完整流程，**示範數字不是研究結果**。改成 `False` 前，請到 cnnp-lab.com/ideas-data 查 IDEAS 的 OpenNeuro 編號，並依 `participants.tsv` 實際欄位修改 `COLS`。

## M0｜環境建置、事先寫定的參數與版本紀錄

In [ ]:
# PyRadiomics 在較新 Python 上若 pip 安裝失敗，改從 GitHub 原始碼安裝
!pip -q install pyradiomics || pip -q install git+https://github.com/AIM-Harvard/pyradiomics.git
!pip -q install SimpleITK nibabel neuroCombat openneuro-py

In [ ]:
import os, json, warnings, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy import stats
from scipy.optimize import brentq
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score, brier_score_loss
warnings.filterwarnings("ignore")

DEMO_MODE = True
SEED = 2026
np.random.seed(SEED); random.seed(SEED)

# ---- 事先寫定（計畫書第四段），分析時不再擴充 ----
TEST_SIZE   = 0.30
ICC_MIN     = 0.75
CORR_MAX    = 0.90
K_MAX       = 5                                  # 訓練集事件 59 ÷ 10，無條件捨去
LASSO_GRID  = np.logspace(-3, 1, 30)             # 30 點
RF_GRID     = {"n_estimators": [300, 500, 1000], "max_depth": [3, 5, None]}   # 9 組
CV          = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
N_BOOT      = 1000
DELTA_AUC_H = 0.05                               # 主要假說：C 比 A 提升 ≥ 0.05

DATASET_ID = "dsXXXXXX"                          # ← IDEAS 的 OpenNeuro 編號
ROOT = "/content/drive/MyDrive/ideas_radiomics" if not DEMO_MODE else "/content/demo_radiomics"
os.makedirs(ROOT, exist_ok=True)
COLS = {"id": "participant_id", "side": "resection_side", "resection": "resection_type",
        "pathology": "pathology", "ilae12": "ilae_12m", "age": "age_group", "sex": "sex"}   # ← 依實際欄位修改
ASEG = {"hip_L": 17, "hip_R": 53, "amy_L": 18, "amy_R": 54}

import sklearn
versions = {"python_seed": SEED, "numpy": np.__version__, "sklearn": sklearn.__version__,
            "SimpleITK": sitk.Version_VersionString()}
try:
    import radiomics; versions["pyradiomics"] = radiomics.__version__
except ImportError:
    versions["pyradiomics"] = "not installed"
json.dump(versions, open(os.path.join(ROOT, "versions.json"), "w"), indent=1)
print(versions)

## M1｜整理世代、納入排除、**先切分**（計畫書步驟 1）

切分只用側別分層，不看任何影像特徵。測試集的病人編號寫入 `test_ids.csv` 後，M4 完全不讀取測試集的特徵。

In [ ]:
if not DEMO_MODE:
    from google.colab import drive; drive.mount("/content/drive")
    import openneuro, glob
    RAW = os.path.join(ROOT, "raw")
    openneuro.download(dataset=DATASET_ID, target_dir=RAW,
                       include=["participants.tsv", "*T1w.nii.gz", "*FLAIR.nii.gz", "*aseg*"])
    part = pd.read_csv(os.path.join(RAW, "participants.tsv"), sep="\t")
    flow = [("IDEAS 病人總數", len(part))]
    # 依計畫書第三段的事先寫定條件逐步排除（欄位內容依實際資料調整）
    side = part[COLS["side"]].astype(str).str.lower()
    part = part[side.isin(["left", "right", "l", "r"])]; flow.append(("單側切除", len(part)))
    part = part[part[COLS["resection"]].astype(str).str.contains("temporal", case=False)]; flow.append(("顳葉切除", len(part)))
    part["y"] = part[COLS["side"]].astype(str).str.lower().str.startswith("r").astype(int)   # 1 = 右側（事件）
    def find_files(pid):   # ← 先用 !find 確認 BIDS 與 derivatives 的實際路徑
        t1 = glob.glob(f"{RAW}/{pid}/**/anat/*T1w.nii.gz", recursive=True)
        fl = glob.glob(f"{RAW}/{pid}/**/anat/*FLAIR.nii.gz", recursive=True)
        sg = glob.glob(f"{RAW}/derivatives/**/{pid}/**/*aseg*", recursive=True)
        return pd.Series([t1[0] if t1 else None, fl[0] if fl else None, sg[0] if sg else None])
    part[["t1", "flair", "aseg"]] = part[COLS["id"]].apply(find_files)
    part = part.dropna(subset=["t1", "aseg"]); flow.append(("具 T1 與分割", len(part)))
    cohort = part.rename(columns={COLS["id"]: "pid"}).set_index("pid")
else:
    # 示範：與計畫書預估相同的人數與側別分布（左 122、右 85）
    rng = np.random.default_rng(SEED)
    n_L, n_R = 122, 85
    cohort = pd.DataFrame({"y": np.r_[np.zeros(n_L), np.ones(n_R)].astype(int)},
                          index=[f"sub-{i:03d}" for i in range(n_L + n_R)])
    flow = [("示範世代", len(cohort))]

train_ids, test_ids = train_test_split(cohort.index, test_size=TEST_SIZE, stratify=cohort["y"], random_state=SEED)
pd.Series(test_ids).to_csv(os.path.join(ROOT, "test_ids.csv"), index=False)
for name, ids in [("全部", cohort.index), ("訓練集", train_ids), ("保留測試集", test_ids)]:
    y = cohort.loc[ids, "y"]
    print(f"{name}: n = {len(ids)}，左 {int((y == 0).sum())}，右（事件）{int(y.sum())}")
print("排除流程：", flow)

## M2｜逐一受試者前處理（計畫書步驟 0）

每個運算只用該受試者自己的影像：N4 偏場校正 → B-spline 重新取樣至 1 mm → 以自己的腦組織平均與標準差做 z 分數（截斷 ±3）。遮罩一律最近鄰內插。

In [ ]:
def n4(img, mask):
    return sitk.N4BiasFieldCorrectionImageFilter().Execute(sitk.Cast(img, sitk.sitkFloat32), mask)

def to_iso(img, is_mask=False, ref=None):
    interp = sitk.sitkNearestNeighbor if is_mask else sitk.sitkBSpline
    if ref is not None:
        return sitk.Resample(img, ref, sitk.Transform(), interp, 0)
    sp = (1.0, 1.0, 1.0)
    size = [int(round(s * o / n)) for s, o, n in zip(img.GetSize(), img.GetSpacing(), sp)]
    return sitk.Resample(img, size, sitk.Transform(), interp, img.GetOrigin(), sp, img.GetDirection(), 0, img.GetPixelID())

def subject_z(img, brain):
    a = sitk.GetArrayFromImage(img).astype(np.float32); m = sitk.GetArrayFromImage(brain) > 0
    z = np.clip((a - a[m].mean()) / a[m].std(), -3, 3)
    out = sitk.GetImageFromArray(z); out.CopyInformation(img); return out

def preprocess(img_path, seg_path):
    img = sitk.ReadImage(img_path)
    seg = to_iso(sitk.ReadImage(seg_path), is_mask=True, ref=img)
    img = n4(img, sitk.Cast(seg > 0, sitk.sitkUInt8))
    img = to_iso(img); seg = to_iso(seg, is_mask=True, ref=img)
    return subject_z(img, sitk.Cast(seg > 0, sitk.sitkUInt8)), seg

## M3｜PyRadiomics 萃取與左右不對稱指標（計畫書步驟 0）

設定與計畫書第四段一致：binWidth = 0.05（z 單位）、僅原始影像、七類特徵；形狀特徵只從 T1 取。每個海馬迴遮罩另做 1 體素侵蝕與膨脹，三個版本都萃取，供 M4 在**訓練摺內**計算 ICC。

In [ ]:
from radiomics import featureextractor
import logging, yaml; logging.getLogger("radiomics").setLevel(logging.ERROR)

PARAMS = {"setting": {"binWidth": 0.05, "normalize": False, "label": 1, "geometryTolerance": 1e-4},
          "imageType": {"Original": {}},
          "featureClass": {k: [] for k in ["shape", "firstorder", "glcm", "glrlm", "glszm", "gldm", "ngtdm"]}}
yaml.safe_dump(PARAMS, open(os.path.join(ROOT, "params.yaml"), "w"))

def extractor(no_shape=False, bin_width=None):
    ex = featureextractor.RadiomicsFeatureExtractor(os.path.join(ROOT, "params.yaml"))
    if no_shape: ex.disableFeatureClassByName("shape")
    if bin_width: ex.settings["binWidth"] = bin_width
    return ex

def mask_versions(seg, label):
    roi = sitk.Cast(seg == label, sitk.sitkUInt8)
    return {"v0": roi, "v1": sitk.BinaryErode(roi, [1, 1, 1]), "v2": sitk.BinaryDilate(roi, [1, 1, 1])}

def feats(img, roi, ex, prefix):
    r = ex.execute(img, roi, label=1)
    return {f"{prefix}_{k[9:]}": float(v) for k, v in r.items() if k.startswith("original_")}

def asymmetry(row_L, row_R):
    out = {}
    for k, L in row_L.items():
        R = row_R[k.replace("hip_L", "hip_R")]; den = (L + R) / 2
        out[k.replace("hip_L", "hipAI")] = (L - R) / den if abs(den) > 1e-6 else L - R
    return out

if DEMO_MODE:
    import radiomics
    ip, mp = radiomics.getTestCase("brain1")
    demo = feats(sitk.ReadImage(ip), sitk.ReadImage(mp), extractor(bin_width=25), "demo")   # 示範影像未 z 標準化
    print(f"示範萃取成功：{len(demo)} 個特徵")
else:
    exT1, exFL = extractor(), extractor(no_shape=True)
    rows = {v: [] for v in ["v0", "v1", "v2"]}
    for pid, r in cohort.iterrows():
        img, seg = preprocess(r["t1"], r["aseg"])
        fl = preprocess(r["flair"], r["aseg"]) if isinstance(r.get("flair"), str) else None
        mv = {s: mask_versions(seg, ASEG[s]) for s in ["hip_L", "hip_R"]}
        for v in rows:
            side = {}
            for s in ["hip_L", "hip_R"]:
                d = feats(img, mv[s][v], exT1, f"T1_{s}")
                if fl is not None:
                    d.update(feats(fl[0], to_iso(mv[s][v], True, fl[0]), exFL, f"FL_{s}"))
                side[s] = d
            rows[v].append({"pid": pid, **asymmetry(side["hip_L"], side["hip_R"])})
    for v in rows:
        pd.DataFrame(rows[v]).set_index("pid").to_csv(os.path.join(ROOT, f"features_AI_{v}.csv"))

## M4｜只用訓練集：摺內建模、選超參數、定 Youden 切點、定版（計畫書步驟 2–4）

`RadiomicsModel` 在每一摺內部依序做 ComBat → ICC（只用該摺訓練受試者）→ 冗餘移除 → 標準化 → LASSO，選出最多 5 個特徵。`GridSearchCV(refit=True)` 在選定超參數後自動以完整訓練集重新擬合一次，即步驟 4 的定版。

In [ ]:
META = ["_batch", "_age", "_sex"]; BASE = ["vol_ai", "_age", "_sex"]

if DEMO_MODE:
    # 模擬 200 個不對稱特徵、三版本遮罩、兩種協定的批次效應、體積不對稱與 HS 病理（僅供演練）
    rng = np.random.default_rng(SEED); n = len(cohort); y = cohort["y"].values; p = 200
    signal = np.zeros(p); signal[:6] = 0.30
    base = rng.normal(0, 1, (n, p)) + np.outer(np.where(y == 1, -1, 1), signal)
    base[:, 6:26] = base[:, [0]] * 0.95 + rng.normal(0, 0.3, (n, 20))
    batch = rng.integers(0, 2, n); base += np.outer(batch * 0.8, rng.normal(1, 0.3, p))
    noise_sd = np.r_[rng.uniform(0.05, 0.3, p - 40), rng.uniform(0.8, 1.5, 40)]       # 後 40 個特徵不穩定
    cols = [f"hipAI_f{i:03d}" for i in range(p)]
    F = {v: pd.DataFrame(base + (0 if v == "v0" else rng.normal(0, 1, (n, p)) * noise_sd), index=cohort.index, columns=cols)
         for v in ["v0", "v1", "v2"]}
    meta = pd.DataFrame({"_batch": batch, "_age": rng.integers(18, 70, n) // 5, "_sex": rng.integers(0, 2, n),
                         "vol_ai": np.where(y == 1, 0.05, -0.05) + rng.normal(0, 0.09, n),
                         "hs": rng.integers(0, 2, n)}, index=cohort.index)
else:
    F = {v: pd.read_csv(os.path.join(ROOT, f"features_AI_{v}.csv"), index_col=0) for v in ["v0", "v1", "v2"]}
    meta = pd.read_csv(os.path.join(ROOT, "meta.csv"), index_col=0)   # _batch（協定群組）、_age、_sex、vol_ai、hs

RAD = list(F["v0"].columns)
X_all = pd.concat([meta[META + ["vol_ai"]], F["v0"]], axis=1)
y_all = cohort["y"]
X_tr, y_tr = X_all.loc[train_ids], y_all.loc[train_ids].values      # M4 只用這兩個
print("訓練集：", X_tr.shape, "事件數 =", int(y_tr.sum()))

In [ ]:
def icc31(M):
    n, k = M.shape; g = M.mean()
    msr = k * ((M.mean(1) - g) ** 2).sum() / (n - 1); msc = n * ((M.mean(0) - g) ** 2).sum() / (k - 1)
    mse = (((M - g) ** 2).sum() - msr * (n - 1) - msc * (k - 1)) / ((n - 1) * (k - 1))
    return (msr - mse) / (msr + (k - 1) * mse)

class RadiomicsModel(ClassifierMixin, BaseEstimator):
    # 摺內：ComBat → ICC → 冗餘移除 → 標準化 → LASSO（≤ K_MAX）→ 最終分類器
    # use_base=True 為模型 C（加上 BASE 變項），False 為模型 B；final='rf' 為 C-RF
    def __init__(self, C=1.0, use_base=True, final="lr", n_estimators=500, max_depth=None):
        self.C, self.use_base, self.final, self.n_estimators, self.max_depth = C, use_base, final, n_estimators, max_depth
    def _transform_rad(self, X):
        Z = X[self.rad_].values
        if self.ok_combat_:
            from neuroCombat import neuroCombatFromTraining
            Z = neuroCombatFromTraining(dat=Z.T, batch=X["_batch"].values, estimates=self.combat_est_)["data"].T
        Z = pd.DataFrame(Z, columns=self.rad_, index=X.index)[self.keep_]
        return self.scaler_.transform(Z)[:, self.sel_idx_]
    def fit(self, X, y):
        y = np.asarray(y); self.classes_ = np.unique(y)
        ids = X.index
        # 1) ICC：只用本摺訓練受試者的三版本特徵
        icc = {c: icc31(np.c_[F["v0"].loc[ids, c], F["v1"].loc[ids, c], F["v2"].loc[ids, c]]) for c in RAD}
        self.rad_ = [c for c in RAD if icc[c] >= ICC_MIN]
        # 2) ComBat：只在本摺估計參數
        self.ok_combat_ = False
        Z = X[self.rad_].values
        try:
            from neuroCombat import neuroCombat
            if X["_batch"].nunique() > 1:
                cov = X[["_batch", "_age", "_sex"]].reset_index(drop=True)
                out = neuroCombat(dat=Z.T, covars=cov, batch_col="_batch", categorical_cols=["_sex"], continuous_cols=["_age"])
                Z, self.combat_est_, self.ok_combat_ = out["data"].T, out["estimates"], True
        except Exception:
            pass
        Z = pd.DataFrame(Z, columns=self.rad_, index=ids)
        # 3) 冗餘移除（保留 ICC 較高者）
        order = sorted(self.rad_, key=lambda c: -icc[c]); rho = Z[order].corr("spearman").abs(); keep = []
        for c in order:
            if all(rho.loc[c, k] <= CORR_MAX for k in keep): keep.append(c)
        self.keep_ = keep
        # 4) 標準化（只用本摺統計量）＋ 5) LASSO 選擇，上限 K_MAX
        self.scaler_ = StandardScaler().fit(Z[keep]); Zs = self.scaler_.transform(Z[keep])
        las = LogisticRegression(penalty="l1", C=self.C, solver="liblinear", class_weight="balanced", max_iter=5000).fit(Zs, y)
        coef = las.coef_[0]; nz = np.flatnonzero(coef)
        self.sel_idx_ = nz[np.argsort(-np.abs(coef[nz]))][:K_MAX] if len(nz) else np.array([int(np.argmax(np.abs(coef)))])
        self.selected_ = [keep[i] for i in self.sel_idx_]
        # 6) 最終分類器
        D = Zs[:, self.sel_idx_]
        if self.use_base:
            self.base_scaler_ = StandardScaler().fit(X[BASE]); D = np.c_[self.base_scaler_.transform(X[BASE]), D]
        self.clf_ = (RandomForestClassifier(n_estimators=self.n_estimators, max_depth=self.max_depth, class_weight="balanced", random_state=SEED)
                     if self.final == "rf" else LogisticRegression(class_weight="balanced", max_iter=5000)).fit(D, y)
        return self
    def predict_proba(self, X):
        D = self._transform_rad(X)
        if self.use_base: D = np.c_[self.base_scaler_.transform(X[BASE]), D]
        return self.clf_.predict_proba(D)
    def predict(self, X): return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

def model_A():   # 基準線：不調參
    from sklearn.compose import ColumnTransformer
    return Pipeline([("pick", ColumnTransformer([("b", "passthrough", BASE)])),
                     ("scale", StandardScaler()), ("clf", LogisticRegression(class_weight="balanced", max_iter=5000))])

def cv_summary(gs):
    i = gs.best_index_; return gs.cv_results_["mean_test_score"][i], gs.cv_results_["std_test_score"][i]

# 步驟 2–4
gs_C = GridSearchCV(RadiomicsModel(use_base=True), {"C": LASSO_GRID}, scoring="roc_auc", cv=CV, n_jobs=-1, refit=True).fit(X_tr, y_tr)
gs_B = GridSearchCV(RadiomicsModel(use_base=False), {"C": LASSO_GRID}, scoring="roc_auc", cv=CV, n_jobs=-1, refit=True).fit(X_tr, y_tr)
gs_RF = GridSearchCV(RadiomicsModel(use_base=True, final="rf", C=gs_C.best_params_["C"]),
                     RF_GRID, scoring="roc_auc", cv=CV, n_jobs=-1, refit=True).fit(X_tr, y_tr)
from sklearn.model_selection import cross_val_score
cvA = cross_val_score(model_A(), X_tr, y_tr, scoring="roc_auc", cv=CV)
final = {"A": model_A().fit(X_tr, y_tr), "B": gs_B.best_estimator_, "C": gs_C.best_estimator_, "C-RF": gs_RF.best_estimator_}
cv_auc = {"A": (cvA.mean(), cvA.std()), "B": cv_summary(gs_B), "C": cv_summary(gs_C), "C-RF": cv_summary(gs_RF)}
for k, (m, s) in cv_auc.items(): print(f"模型 {k}：訓練集 CV AUC = {m:.3f} ± {s:.3f}")
print("模型 C 選入特徵：", final["C"].selected_)

# 步驟 3：Youden 切點（訓練集樣本外預測，不看測試集）
thr = {}
for k, est in final.items():
    oof = cross_val_predict(clone(est), X_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    f, t, th = roc_curve(y_tr, oof); thr[k] = th[np.argmax(t - f)]
print("Youden 切點：", {k: round(v, 3) for k, v in thr.items()})

# 生物學關聯（RQS 第 7 項）：訓練集內，選入特徵在 HS / 非 HS 間的差異，FDR 校正
from scipy.stats import mannwhitneyu
hs = meta.loc[train_ids, "hs"].values; pv = []
for c in final["C"].selected_:
    v = X_tr[c].values; pv.append(mannwhitneyu(v[hs == 1], v[hs == 0]).pvalue)
pv = np.array(pv); order = np.argsort(pv); m = len(pv)
q = np.minimum.accumulate((pv[order] * m / np.arange(1, m + 1))[::-1])[::-1]       # Benjamini–Hochberg
bio = pd.DataFrame({"feature": np.array(final["C"].selected_)[order], "p": pv[order], "q_FDR": np.minimum(q, 1)})
print(bio.round(4))

## M5｜保留測試集只開啟一次（計畫書步驟 5–6）

`open_test_once()` 執行第二次會直接報錯。若在這裡發現問題，**記錄為限制，不回頭修改 M4**。

In [ ]:
_TEST_OPENED = False
def open_test_once():
    global _TEST_OPENED
    if _TEST_OPENED: raise RuntimeError("保留測試集已開啟過；若需再次開啟，必須在報告中揭露並標示為探索性。")
    _TEST_OPENED = True
    return X_all.loc[test_ids], y_all.loc[test_ids].values

def boot(fn, y, *ps, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); out = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2: out.append(fn(y[i], *[p[i] for p in ps]))
    return np.percentile(out, [2.5, 97.5])

def delong(y, p1, p2):
    def comp(p):
        pos, neg = p[y == 1], p[y == 0]
        return (np.array([np.mean((a > neg) + 0.5 * (a == neg)) for a in pos]),
                np.array([np.mean((pos > b) + 0.5 * (pos == b)) for b in neg]))
    a10, a01 = comp(p1); b10, b01 = comp(p2)
    S = np.cov(np.vstack([a10, b10])) / len(a10) + np.cov(np.vstack([a01, b01])) / len(a01)
    d = a10.mean() - b10.mean(); z = d / np.sqrt(S[0, 0] + S[1, 1] - 2 * S[0, 1])
    return d, 2 * stats.norm.sf(abs(z))

def calib(y, p):
    lp = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    slope = LogisticRegression(C=1e6).fit(lp.reshape(-1, 1), y).coef_[0, 0]
    inter = brentq(lambda a: np.sum(y - 1 / (1 + np.exp(-(a + lp)))), -10, 10)
    return inter, slope

def net_benefit(y, p, ths):
    return np.array([np.sum((p >= t) & (y == 1)) / len(y) - np.sum((p >= t) & (y == 0)) / len(y) * t / (1 - t) for t in ths])

X_te, y_te = open_test_once()
P = {k: m.predict_proba(X_te)[:, 1] for k, m in final.items()}
rows = []
for k, p in P.items():
    lo, hi = boot(roc_auc_score, y_te, p); inter, slope = calib(y_te, p); pred = p >= thr[k]
    rows.append({"模型": k, "測試 AUC": roc_auc_score(y_te, p), "95% CI": f"{lo:.3f}–{hi:.3f}",
                 "AUPRC": average_precision_score(y_te, p), "敏感度": pred[y_te == 1].mean(), "特異度": (~pred[y_te == 0]).mean(),
                 "校準截距": inter, "校準斜率": slope, "Brier": brier_score_loss(y_te, p),
                 "訓練 CV AUC": cv_auc[k][0], "CV−測試 差距": cv_auc[k][0] - roc_auc_score(y_te, p)})
res = pd.DataFrame(rows).round(3); display(res)

# 主要比較（唯一一組）：模型 C vs 模型 A
d, pv = delong(y_te, P["C"], P["A"])
ci = boot(lambda yy, a, b: roc_auc_score(yy, a) - roc_auc_score(yy, b), y_te, P["C"], P["A"])
gap = abs(cv_auc["C"][0] - roc_auc_score(y_te, P["C"]))
print(f"主要比較 ΔAUC(C − A) = {d:.3f}，95% CI {ci[0]:.3f}–{ci[1]:.3f}，DeLong P = {pv:.4f}")
print(f"假說判定：ΔAUC ≥ {DELTA_AUC_H}？{'是' if d >= DELTA_AUC_H else '否'}；CV 與測試差距 < 0.05？{'是' if gap < 0.05 else '否'}（{gap:.3f}）")
print("其餘比較（B、C-RF、子群、跨協定、杏仁核）一律標示為探索性。")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for k, p in P.items():
    f, t, _ = roc_curve(y_te, p); ax[0].plot(f, t, label=f"{k} ({roc_auc_score(y_te, p):.2f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].set(title="Held-out test ROC (opened once)", xlabel="1 - specificity", ylabel="sensitivity"); ax[0].legend()
from sklearn.calibration import calibration_curve
for k in ["A", "C"]:
    fr, mp = calibration_curve(y_te, P[k], n_bins=5, strategy="quantile"); ax[1].plot(mp, fr, "o-", label=k)
ax[1].plot([0, 1], [0, 1], "k--", lw=.8); ax[1].set(title="Calibration (test)", xlabel="predicted", ylabel="observed"); ax[1].legend()
ths = np.linspace(.05, .9, 40); prev = y_te.mean()
for k in ["A", "C"]: ax[2].plot(ths, net_benefit(y_te, P[k], ths), label=k)
ax[2].plot(ths, prev - (1 - prev) * ths / (1 - ths), "k:", label="treat all"); ax[2].axhline(0, color="k", lw=.8)
ax[2].set(title="Decision curve (test)", xlabel="threshold", ylabel="net benefit", ylim=(-.05, prev + .05)); ax[2].legend()
plt.tight_layout(); plt.savefig(os.path.join(ROOT, "fig_test.png"), dpi=200); plt.show()
res.to_csv(os.path.join(ROOT, "test_results.csv"), index=False)
pd.Series(final["C"].selected_).to_csv(os.path.join(ROOT, "selected_features_C.csv"), index=False)

## 切換到真實資料前的檢查清單

1. M0：`DEMO_MODE = False`、填入 `DATASET_ID`、修改 `COLS`。
2. M1：以 `!find` 確認檔案路徑；核對排除流程人數，填入計畫書第三段的「待確認」欄位並凍結。
3. M2：抽 10% 受試者輸出遮罩疊圖目視檢查。
4. M3：完成後另存 `meta.csv`（協定群組 `_batch` 由 BIDS JSON 參數分群、`_age`、`_sex`、`vol_ai`、`hs`）。
5. M4 只執行一次完整搜尋；搜尋範圍不得擴充。
6. M5 只執行一次；結果不論好壞都報告。